In [1]:
!nvidia-smi
!pip install -q torch-geometric mendeleev
!mkdir -p /content/work/out
!if [ ! -d /content/work/FAENet ]; then git clone https://github.com/Gabriel-PEZ/FAENet.git /content/work/FAENet; else git -C /content/work/FAENet pull; fi

Fri Nov 27 11:08:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   67C    P8             15W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# %%
from pathlib import Path

import torch
from torch import nn
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader

print(Path.cwd())
import sys
sys.path.insert(0, "/content/work/FAENet/src")
from faenet.atomic_features import build_atomic_features
from faenet.faenet import FAENet
from torch_geometric.nn import global_mean_pool
from faenet.graph_creation import SFA, adjacency_matrix, rho1_inv

# %%

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
dataset = QM9(root="/content/work/data/qm9")

# %%

atomic_numbers_to_indices, properties_atomes_scaled = build_atomic_features(dataset)
print(atomic_numbers_to_indices)
print(properties_atomes_scaled)

# %%

torch.manual_seed(7)
faenet = FAENet(
    atom_embedding_dim=128,
    hidden_dim_node=400,
    hidden_dim_edge=480,
    hidden_dim_output_block=200,
    node_dim=400,
    edge_dim=480,
    atomic_numbers_to_indices=atomic_numbers_to_indices,
    properties_atomes_scaled=properties_atomes_scaled,
    nb_gaussians_rbf=100,
    cutoff=6.0,
    num_iteration_block=5,
    jump_connection=True,
    frame_averaging="sfa",
).to(device)
faenet.output_block.weight_mlp.append(nn.Sigmoid())
# afin d'éviter le collapse, on remplace l'innit Kaiming par l'innit Xavier que les auteurs utilisent dans leur github

# layers du sequential MLP de 2000 200 à 200 1
# le [1] est l'activation dans le sequential
for layer in [faenet.output_block.mlp[0], faenet.output_block.mlp[2]]:
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)

# ici c'est un linear (400 à 1) car on prend que H5
for layer in [faenet.output_block.weight_mlp[0]]:
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)

with torch.no_grad():
    faenet.output_block.weight_mlp[0].weight.zero_()
    faenet.output_block.weight_mlp[0].bias.fill_(1.0)
faenet.output_block.weight_mlp[0].weight.requires_grad_(False)
faenet.output_block.weight_mlp[0].bias.requires_grad_(False)

# %% Dataset split

idx_regression_target = 0  # Dipole moment mu

split_generator = torch.Generator().manual_seed(2003)
indices = torch.randperm(len(dataset), generator=split_generator)

n_train = 13_750 
n_val = 2000

dataset_train = dataset[indices[:n_train]]
dataset_val = dataset[indices[110_000 : 110_000 + n_val]]
dataset_test = dataset[indices[120_000:]]

# norm target sur train 
target_train = dataset_train.y[:, idx_regression_target].to(device)
mu_target_train = target_train.mean()
std_target_train = target_train.std()

# DataLoaders
shuffle_generator = torch.Generator().manual_seed(2007)

train_loader = DataLoader(
    dataset_train,
    batch_size=64,
    shuffle=True,
    generator=shuffle_generator
)
val_loader = DataLoader(dataset_val, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset_test, batch_size=64, shuffle=False)

# %% 

checkpoint_dir = Path("/content/work/out") / (
    f"qm9_{idx_regression_target}_{faenet.frame_averaging}_fixed_clip"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# %%
# par step d'optimization on entend un batch, car on met à jour tous les poids via backprop sur un batch (64 molécules) = un step
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW([p for p in faenet.parameters() if p.requires_grad], lr=1e-3)
lr_initial = 1e-3
steps_warmup = 500 # 3000/8 = 500 batchs vs les auteurs. ça fait 2 epochs
# où est ce qu'on en est en termes de step pour mettre à jour le LR
global_step = 0

# on crée le scheduler puis on lui passera la val_mae via scheduler(val_mae) pour qu'il divise le LR si la val ne s'améliore pas 
# min car loss
# patience : nombre d'epochs sans amélioration
# on divise par 2 le LR
# plus faible que les auteurs car on entraine sur 100 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer = optimizer, mode = "min", patience = 5, factor = 0.5, min_lr=1e-5)
train_generator = torch.Generator().manual_seed(1977)

num_epochs = 100  # First experiment
best_val_mae = float("inf")

# %% 

for epoch in range(num_epochs):

    # active le mode train
    faenet.train()
    train_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        # on reinitialize les gradients enregistrès du backward d'avant
        optimizer.zero_grad()

        # recup les targets et les normalize
        y_true = (
            batch.y[:, idx_regression_target:idx_regression_target + 1]
            - mu_target_train
        ) / std_target_train

        # predictions pour loss
        _, y_pred = faenet(batch, sfa_generator=train_generator)

        assert y_true.shape == y_pred.shape
        
        loss = criterion(y_pred, y_true)
        # calcule les gradients
        loss.backward()

        torch.nn.utils.clip_grad_norm_(faenet.parameters(), max_norm=5.0)
        # warmup -> augmenter progressivement le LR vers sa valeur initiale
        if global_step < steps_warmup:
            # facteur d'avancement du warmup
            lr = lr_initial * (global_step + 1) / steps_warmup
            # les HP de Adam
            for pg in optimizer.param_groups:
                # le LR
                pg["lr"] = lr

        

        optimizer.step()
        global_step += 1
        train_loss += loss.item() * batch.num_graphs

    train_loss /= len(dataset_train)

    # val
    faenet.eval()
    val_mae = 0.0
    val_generator = torch.Generator().manual_seed(1972)

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)

            _, y_pred_scaled = faenet(
                batch, sfa_generator=val_generator
            )

            y_pred = mu_target_train + std_target_train * y_pred_scaled
            y_true = batch.y[:, idx_regression_target:idx_regression_target + 1]

            val_mae += (y_pred - y_true).abs().sum().item()

    val_mae /= len(dataset_val)
    # on initialise le scheduler
    if global_step > steps_warmup:
        scheduler.step(val_mae)
    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Val MAE: {val_mae:.6f}"
    )

    # Checkpoints
    is_best = val_mae < best_val_mae
    best_val_mae = min(best_val_mae, val_mae)

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": faenet.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_mae": best_val_mae,
        "frame_averaging": faenet.frame_averaging,
        "target_idx": idx_regression_target,
        "target_mean": mu_target_train.detach().cpu(),
        "target_std": std_target_train.detach().cpu(),
        "sfa_generator_state": train_generator.get_state(),
        "shuffle_generator_state": shuffle_generator.get_state(),
        "torch_rng_state": torch.get_rng_state(),
    }

    torch.save(checkpoint, checkpoint_dir / "last.pt")

    if is_best:
        torch.save(checkpoint, checkpoint_dir / "best.pt")


    # check anti collapse du MLP de output block
    # on reconstruit toute la chaine et on regarde le nombre de neurones actifs pour le block d'output
    # et à quel point le modèle utilise les neurones vs juste le biais (prédiction constante)
    with torch.no_grad():
        b0 = next(iter(val_loader)).to(device)
        H = faenet.molecule_embedding(b0)
        U0 = SFA(b0.pos, batch_indices=b0.batch, full=True)[0]
        t0 = global_mean_pool(b0.pos, b0.batch)
        X_loc = rho1_inv(b0.pos, t0, U0, b0.batch)
        _, _, ei = adjacency_matrix(X_loc, faenet.cutoff, b0.batch)
        E = faenet.edge_embedding(X_loc, ei)
        Hs = []
        for blk in faenet.list_interaction_blocks:
            H = blk(H, E, ei, b0.batch)
            Hs.append(H)
        pre = faenet.output_block.mlp[0](torch.cat(Hs, dim=1))
        y_at = faenet.output_block.mlp[2](faenet.output_block.mlp[1](pre))
        bias = faenet.output_block.mlp[2].bias.item()
        active_neurons = (pre.max(dim=0).values > -5.0).sum().item()
        max_y_dev = (y_at - bias).abs().max().item()
    print(f"active neurons {active_neurons}")
    print(f"vs only bias {max_y_dev}")
# %%

checkpoint = torch.load(checkpoint_dir / "best.pt", map_location=device, weights_only=True)
faenet.load_state_dict(checkpoint["model_state_dict"])
faenet.frame_averaging = checkpoint["frame_averaging"]
# faenet.frame_averaging = "full" check l'effet du full fa en inférence
faenet.eval()

# norm du training
mu = checkpoint["target_mean"].to(device)
std = checkpoint["target_std"].to(device)
target_idx = checkpoint["target_idx"]

test_generator = torch.Generator().manual_seed(1972)

test_mae = 0.0
n_samples = 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        _, y_pred_scaled = faenet(
            batch, sfa_generator=test_generator
        )

        y_pred = mu + std * y_pred_scaled
        y_true = batch.y[:, target_idx:target_idx + 1]

        test_mae += (y_pred - y_true).abs().sum().item()
        n_samples += y_true.shape[0]

test_mae /= n_samples

print(f"Best epoch: {checkpoint['epoch']}")
print(f"Best Val MAE: {checkpoint['best_val_mae']:.6f}")
print(f"Test MAE: {test_mae:.6f}")

# %%


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


/content
cuda


Extracting /content/work/data/qm9/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


{1: [0, 0, 0], 6: [1, 1, 1], 7: [2, 2, 1], 8: [3, 3, 1], 9: [4, 4, 1]}
tensor([[-1.8196,  0.5092, -0.5011, -0.7153, -0.2221, -1.2166, -1.8461, -1.9122,
         -0.2455,  0.0000],
        [ 1.0039, -2.0000,  2.0000,  1.7863,  0.1082, -0.8648,  1.1638,  0.9231,
         -1.4212, -1.4981],
        [ 0.6902,  0.4970, -0.4999,  0.3501, -1.6231, -0.1124,  0.4114,  0.6594,
          0.2250, -0.3201],
        [ 0.3765,  0.4967, -0.4997, -0.4233,  0.2376,  0.6767,  0.2609,  0.1319,
         -0.2357,  0.9260],
        [-0.2510,  0.4971, -0.4994, -0.9978,  1.4994,  1.5171,  0.0100,  0.1978,
          1.6774,  0.8921]])
Epoch 1/100 | Train MSE: 5.897602 | Val MAE: 1.314176
active neurons 200
vs only bias 0.8838287591934204
Epoch 2/100 | Train MSE: 2.265110 | Val MAE: 1.311420
active neurons 199
vs only bias 0.44331589341163635
Epoch 3/100 | Train MSE: 0.748660 | Val MAE: 0.985820
active neurons 153
vs only bias 0.6966413855552673
Epoch 4/100 | Train MSE: 0.582896 | Val MAE: 0.822134
active neuron

: 

In [ ]:

checkpoint = torch.load(checkpoint_dir / "best.pt", map_location=device, weights_only=True)
faenet.load_state_dict(checkpoint["model_state_dict"])
faenet.frame_averaging = checkpoint["frame_averaging"]
# faenet.frame_averaging = "full" check l'effet du full fa en inférence
faenet.eval()

# norm du training
mu = checkpoint["target_mean"].to(device)
std = checkpoint["target_std"].to(device)
target_idx = checkpoint["target_idx"]

test_generator = torch.Generator().manual_seed(1972)

test_mae = 0.0
n_samples = 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        _, y_pred_scaled = faenet(
            batch, sfa_generator=test_generator
        )

        y_pred = mu + std * y_pred_scaled
        y_true = batch.y[:, target_idx:target_idx + 1]

        test_mae += (y_pred - y_true).abs().sum().item()
        n_samples += y_true.shape[0]

test_mae /= n_samples

print(f"Best epoch: {checkpoint['epoch']}")
print(f"Best Val MAE: {checkpoint['best_val_mae']:.6f}")
print(f"Test MAE: {test_mae:.6f}")

# %%


Best epoch: 61
Best Val MAE: 0.281911
Test MAE: 0.273828


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import shutil

drive_root = Path("/content/drive/MyDrive/faenet_runs")
drive_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# %%
from pathlib import Path
import shutil
import traceback

import torch
from torch import nn
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader

print(Path.cwd())
import sys
sys.path.insert(0, "/content/work/FAENet/src")
from faenet.atomic_features import build_atomic_features
from faenet.faenet import FAENet
from torch_geometric.nn import global_mean_pool
from faenet.graph_creation import SFA, adjacency_matrix, rho1_inv

# %%

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
dataset = QM9(root="/content/work/data/qm9")

# %%

atomic_numbers_to_indices, properties_atomes_scaled = build_atomic_features(dataset)
print(atomic_numbers_to_indices)
print(properties_atomes_scaled)

# %% Dataset split (commun à tous les modes)

idx_regression_target = 0  # Dipole moment mu

split_generator = torch.Generator().manual_seed(2003)
indices = torch.randperm(len(dataset), generator=split_generator)

n_train = 13_750
n_val = 2000

dataset_train = dataset[indices[:n_train]]
dataset_val = dataset[indices[110_000 : 110_000 + n_val]]
dataset_test = dataset[indices[120_000:]]

# norm target sur train
target_train = dataset_train.y[:, idx_regression_target].to(device)
mu_target_train = target_train.mean()
std_target_train = target_train.std()

drive_root = Path("/content/drive/MyDrive/faenet_runs")
drive_root.mkdir(parents=True, exist_ok=True)

# %% Boucle sur les modes (none en premier car rapide, full en second)

for mode in ["none", "full"]:
    print(f"\n========== START MODE: {mode} ==========")
    checkpoint_dir = drive_root / f"qm9_{idx_regression_target}_{mode}_fixed_clip"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_path = checkpoint_dir / "log.txt"

    try:
        torch.manual_seed(7)
        faenet = FAENet(
            atom_embedding_dim=128,
            hidden_dim_node=400,
            hidden_dim_edge=480,
            hidden_dim_output_block=200,
            node_dim=400,
            edge_dim=480,
            atomic_numbers_to_indices=atomic_numbers_to_indices,
            properties_atomes_scaled=properties_atomes_scaled,
            nb_gaussians_rbf=100,
            cutoff=6.0,
            num_iteration_block=5,
            jump_connection=True,
            frame_averaging=mode,
        ).to(device)

        # Init Xavier sur mlp
        for layer in [faenet.output_block.mlp[0], faenet.output_block.mlp[2]]:
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

        # Option 1a : figer weight_mlp à 1.0 (sans Sigmoid)
        with torch.no_grad():
            faenet.output_block.weight_mlp[0].weight.zero_()
            faenet.output_block.weight_mlp[0].bias.fill_(1.0)
        faenet.output_block.weight_mlp[0].weight.requires_grad_(False)
        faenet.output_block.weight_mlp[0].bias.requires_grad_(False)

        # Batch size / grad accum (pour éviter OOM en mode full sur T4)
        accum_steps = 4 if mode == "full" else 1
        batch_size = 64 // accum_steps

        shuffle_generator = torch.Generator().manual_seed(2007)
        train_loader = DataLoader(
            dataset_train,
            batch_size=batch_size,
            shuffle=True,
            generator=shuffle_generator,
        )
        val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(
            [p for p in faenet.parameters() if p.requires_grad], lr=1e-3
        )
        lr_initial = 1e-3
        steps_warmup = 500
        global_step = 0

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=optimizer, mode="min", patience=5, factor=0.5, min_lr=1e-5
        )
        train_generator = torch.Generator().manual_seed(1977)

        num_epochs = 80 if mode == "full" else 100
        best_val_mae = float("inf")

        for epoch in range(num_epochs):
            faenet.train()
            train_loss = 0.0
            optimizer.zero_grad()

            for step, batch in enumerate(train_loader):
                batch = batch.to(device)

                y_true = (
                    batch.y[:, idx_regression_target:idx_regression_target + 1]
                    - mu_target_train
                ) / std_target_train

                _, y_pred = faenet(batch, sfa_generator=train_generator)
                assert y_true.shape == y_pred.shape

                loss = criterion(y_pred, y_true)
                (loss / accum_steps).backward()

                if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(faenet.parameters(), max_norm=5.0)
                    if global_step < steps_warmup:
                        lr = lr_initial * (global_step + 1) / steps_warmup
                        for pg in optimizer.param_groups:
                            pg["lr"] = lr
                    optimizer.step()
                    optimizer.zero_grad()
                    global_step += 1

                train_loss += loss.item() * batch.num_graphs

            train_loss /= len(dataset_train)

            # val
            faenet.eval()
            val_mae = 0.0
            val_generator = torch.Generator().manual_seed(1972)

            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)
                    _, y_pred_scaled = faenet(batch, sfa_generator=val_generator)
                    y_pred = mu_target_train + std_target_train * y_pred_scaled
                    y_true = batch.y[:, idx_regression_target:idx_regression_target + 1]
                    val_mae += (y_pred - y_true).abs().sum().item()

            val_mae /= len(dataset_val)
            if global_step > steps_warmup:
                scheduler.step(val_mae)

            # check anti collapse
            with torch.no_grad():
                b0 = next(iter(val_loader)).to(device)
                H = faenet.molecule_embedding(b0)
                if mode == "none":
                    X_loc = b0.pos
                else:
                    U0 = SFA(b0.pos, batch_indices=b0.batch, full=True)[0]
                    t0 = global_mean_pool(b0.pos, b0.batch)
                    X_loc = rho1_inv(b0.pos, t0, U0, b0.batch)
                _, _, ei = adjacency_matrix(X_loc, faenet.cutoff, b0.batch)
                E = faenet.edge_embedding(X_loc, ei)
                Hs = []
                for blk in faenet.list_interaction_blocks:
                    H = blk(H, E, ei, b0.batch)
                    Hs.append(H)
                pre = faenet.output_block.mlp[0](torch.cat(Hs, dim=1))
                y_at = faenet.output_block.mlp[2](faenet.output_block.mlp[1](pre))
                bias = faenet.output_block.mlp[2].bias.item()
                active_neurons = (pre.max(dim=0).values > -5.0).sum().item()
                max_y_dev = (y_at - bias).abs().max().item()

            msg = (
                f"[{mode}] Epoch {epoch + 1}/{num_epochs} | "
                f"Train MSE: {train_loss:.6f} | Val MAE: {val_mae:.6f} | "
                f"active: {active_neurons} | vs_bias: {max_y_dev:.4f}"
            )
            print(msg)
            with open(log_path, "a") as f:
                f.write(msg + "\n")

            # Checkpoints (écriture locale puis copie sur Drive)
            is_best = val_mae < best_val_mae
            best_val_mae = min(best_val_mae, val_mae)

            checkpoint = {
                "epoch": epoch + 1,
                "model_state_dict": faenet.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_mae": best_val_mae,
                "frame_averaging": faenet.frame_averaging,
                "target_idx": idx_regression_target,
                "target_mean": mu_target_train.detach().cpu(),
                "target_std": std_target_train.detach().cpu(),
                "sfa_generator_state": train_generator.get_state(),
                "shuffle_generator_state": shuffle_generator.get_state(),
                "torch_rng_state": torch.get_rng_state(),
            }

            torch.save(checkpoint, "/content/tmp_last.pt")
            shutil.copy("/content/tmp_last.pt", checkpoint_dir / "last.pt")

            if is_best:
                torch.save(checkpoint, "/content/tmp_best.pt")
                shutil.copy("/content/tmp_best.pt", checkpoint_dir / "best.pt")

        # Eval test sur best.pt
        checkpoint = torch.load(checkpoint_dir / "best.pt", map_location=device, weights_only=True)
        faenet.load_state_dict(checkpoint["model_state_dict"])
        faenet.frame_averaging = checkpoint["frame_averaging"]
        faenet.eval()

        mu = checkpoint["target_mean"].to(device)
        std = checkpoint["target_std"].to(device)
        target_idx = checkpoint["target_idx"]
        test_generator = torch.Generator().manual_seed(1972)

        test_mae = 0.0
        n_samples = 0
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                _, y_pred_scaled = faenet(batch, sfa_generator=test_generator)
                y_pred = mu + std * y_pred_scaled
                y_true = batch.y[:, target_idx:target_idx + 1]
                test_mae += (y_pred - y_true).abs().sum().item()
                n_samples += y_true.shape[0]

        test_mae /= n_samples
        summary = (
            f"[{mode}] DONE | Best epoch: {checkpoint['epoch']} | "
            f"Best Val MAE: {checkpoint['best_val_mae']:.6f} | Test MAE: {test_mae:.6f}"
        )
        print(summary)
        with open(log_path, "a") as f:
            f.write(summary + "\n")

    except Exception:
        err = traceback.format_exc()
        print(err)
        with open(checkpoint_dir / "error.txt", "w") as f:
            f.write(err)
    finally:
        for var in ["faenet", "optimizer", "scheduler", "train_loader", "val_loader", "test_loader"]:
            if var in locals():
                del locals()[var]
        torch.cuda.empty_cache()

import time
from google.colab import drive, runtime

time.sleep(30)  # laisse le temps à Drive de finir d'écrire les derniers fichiers
try:
    drive.flush_and_unmount()
except Exception as e:
    print("flush_and_unmount warning:", e)

runtime.unassign()  # libère et supprime le serveur Colab

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


/content
cuda
{1: [0, 0, 0], 6: [1, 1, 1], 7: [2, 2, 1], 8: [3, 3, 1], 9: [4, 4, 1]}
tensor([[-1.8196,  0.5092, -0.5011, -0.7153, -0.2221, -1.2166, -1.8461, -1.9122,
         -0.2455,  0.0000],
        [ 1.0039, -2.0000,  2.0000,  1.7863,  0.1082, -0.8648,  1.1638,  0.9231,
         -1.4212, -1.4981],
        [ 0.6902,  0.4970, -0.4999,  0.3501, -1.6231, -0.1124,  0.4114,  0.6594,
          0.2250, -0.3201],
        [ 0.3765,  0.4967, -0.4997, -0.4233,  0.2376,  0.6767,  0.2609,  0.1319,
         -0.2357,  0.9260],
        [-0.2510,  0.4971, -0.4994, -0.9978,  1.4994,  1.5171,  0.0100,  0.1978,
          1.6774,  0.8921]])

========== START MODE: none ==========
[none] Epoch 1/100 | Train MSE: 10.632955 | Val MAE: 4.068620 | active: 200 | vs_bias: 0.6290
[none] Epoch 2/100 | Train MSE: 1.773432 | Val MAE: 1.243818 | active: 200 | vs_bias: 0.6137
[none] Epoch 3/100 | Train MSE: 1.788719 | Val MAE: 0.990188 | active: 107 | vs_bias: 0.9460
[none] Epoch 4/100 | Train MSE: 0.632261 | Val MA

In [ ]:
# %%
from pathlib import Path
import shutil
import traceback

import torch
from torch import nn
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader

print(Path.cwd())
import sys
sys.path.insert(0, "/content/work/FAENet/src")
from faenet.atomic_features import build_atomic_features
from faenet.faenet import FAENet
from torch_geometric.nn import global_mean_pool
from faenet.graph_creation import SFA, adjacency_matrix, rho1_inv

# %%

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
dataset = QM9(root="/content/work/data/qm9")

# %%

atomic_numbers_to_indices, properties_atomes_scaled = build_atomic_features(dataset)
print(atomic_numbers_to_indices)
print(properties_atomes_scaled)

# %% Dataset split

idx_regression_target = 0  # Dipole moment mu

split_generator = torch.Generator().manual_seed(2003)
indices = torch.randperm(len(dataset), generator=split_generator)

n_train = 13_750
n_val = 2000

dataset_train = dataset[indices[:n_train]]
dataset_val = dataset[indices[110_000 : 110_000 + n_val]]
dataset_test = dataset[indices[120_000:]]

# norm target sur train
target_train = dataset_train.y[:, idx_regression_target].to(device)
mu_target_train = target_train.mean()
std_target_train = target_train.std()

drive_root = Path("/content/drive/MyDrive/faenet_runs")
drive_root.mkdir(parents=True, exist_ok=True)

# %% Boucle sur les modes (mets ["sfa"] si tu veux relancer juste sfa, ou ["sfa", "none", "full"])

modes = ["sfa", "none", "full"]

for mode in modes:
    print(f"\n========== START MODE: {mode} ==========")
    checkpoint_dir = drive_root / f"qm9_{idx_regression_target}_{mode}_noweights"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_path = checkpoint_dir / "log.txt"

    try:
        torch.manual_seed(7)
        faenet = FAENet(
            atom_embedding_dim=128,
            hidden_dim_node=400,
            hidden_dim_edge=480,
            hidden_dim_output_block=200,
            node_dim=400,
            edge_dim=480,
            atomic_numbers_to_indices=atomic_numbers_to_indices,
            properties_atomes_scaled=properties_atomes_scaled,
            nb_gaussians_rbf=100,
            cutoff=6.0,
            num_iteration_block=5,
            jump_connection=True,
            frame_averaging=mode,
            use_weights=False,
            xavier_init=True,
        ).to(device)

        # DataLoaders (accum_steps=4 en full pour garder batch effectif=64 sans OOM sur T4)
        accum_steps = 1 if mode == "full" else 1
        batch_size = 64 // accum_steps
        shuffle_generator = torch.Generator().manual_seed(2007)

        train_loader = DataLoader(
            dataset_train,
            batch_size=batch_size,
            shuffle=True,
            generator=shuffle_generator,
        )
        val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

        # par step d'optimization on entend un batch effectif (64 molécules) = un step
        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(faenet.parameters(), lr=1e-3)
        lr_initial = 1e-3
        steps_warmup = 500  # 3000/8 = 500 batchs vs les auteurs. ça fait 2 epochs
        # où est ce qu'on en est en termes de step pour mettre à jour le LR
        global_step = 0

        # on crée le scheduler puis on lui passera la val_mae via scheduler.step(val_mae) pour qu'il divise le LR si la val ne s'améliore pas
        # min car loss
        # patience : nombre d'epochs sans amélioration
        # on divise par 2 le LR
        # plus faible que les auteurs car on entraine sur 100 epochs
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=optimizer, mode="min", patience=5, factor=0.5, min_lr=1e-5
        )
        train_generator = torch.Generator().manual_seed(1977)

        num_epochs = 80 if mode == "full" else 100
        best_val_mae = float("inf")

        for epoch in range(num_epochs):

            # active le mode train
            faenet.train()
            train_loss = 0.0
            # on reinitialize les gradients enregistrés du backward d'avant
            optimizer.zero_grad()

            for step, batch in enumerate(train_loader):
                batch = batch.to(device)

                # recup les targets et les normalize
                y_true = (
                    batch.y[:, idx_regression_target:idx_regression_target + 1]
                    - mu_target_train
                ) / std_target_train

                # predictions pour loss
                _, y_pred = faenet(batch, sfa_generator=train_generator)

                assert y_true.shape == y_pred.shape

                loss = criterion(y_pred, y_true)
                # calcule les gradients
                (loss / accum_steps).backward()

                if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(faenet.parameters(), max_norm=5.0)
                    # warmup -> augmenter progressivement le LR vers sa valeur initiale
                    if global_step < steps_warmup:
                        # facteur d'avancement du warmup
                        lr = lr_initial * (global_step + 1) / steps_warmup
                        # les HP de Adam
                        for pg in optimizer.param_groups:
                            # le LR
                            pg["lr"] = lr

                    optimizer.step()
                    optimizer.zero_grad()
                    global_step += 1

                train_loss += loss.item() * batch.num_graphs

            train_loss /= len(dataset_train)

            # val
            faenet.eval()
            val_mae = 0.0
            val_generator = torch.Generator().manual_seed(1972)

            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)

                    _, y_pred_scaled = faenet(
                        batch, sfa_generator=val_generator
                    )

                    y_pred = mu_target_train + std_target_train * y_pred_scaled
                    y_true = batch.y[:, idx_regression_target:idx_regression_target + 1]

                    val_mae += (y_pred - y_true).abs().sum().item()

            val_mae /= len(dataset_val)
            # on met à jour le scheduler après le warmup
            if global_step > steps_warmup:
                scheduler.step(val_mae)

            # check anti collapse du MLP de output block
            # on reconstruit toute la chaine et on regarde le nombre de neurones actifs pour le block d'output
            # et à quel point le modèle utilise les neurones vs juste le biais (prédiction constante)
            with torch.no_grad():
                b0 = next(iter(val_loader)).to(device)
                H = faenet.molecule_embedding(b0)
                if mode == "none":
                    X_loc = b0.pos
                else:
                    U0 = SFA(b0.pos, batch_indices=b0.batch, full=True)[0]
                    t0 = global_mean_pool(b0.pos, b0.batch)
                    X_loc = rho1_inv(b0.pos, t0, U0, b0.batch)
                _, _, ei = adjacency_matrix(X_loc, faenet.cutoff, b0.batch)
                E = faenet.edge_embedding(X_loc, ei)
                Hs = []
                for blk in faenet.list_interaction_blocks:
                    H = blk(H, E, ei, b0.batch)
                    Hs.append(H)
                H_in = torch.cat(Hs, dim=1) if faenet.jump_connection else H
                pre = faenet.output_block.mlp[0](H_in)
                y_at = faenet.output_block.mlp[2](faenet.output_block.mlp[1](pre))
                bias = faenet.output_block.mlp[2].bias.item()
                active_neurons = (pre.max(dim=0).values > -5.0).sum().item()
                max_y_dev = (y_at - bias).abs().max().item()

            msg = (
                f"[{mode}] Epoch {epoch + 1}/{num_epochs} | "
                f"Train MSE: {train_loss:.6f} | Val MAE: {val_mae:.6f} | "
                f"active: {active_neurons} | vs_bias: {max_y_dev:.4f}"
            )
            print(msg)
            with open(log_path, "a") as f:
                f.write(msg + "\n")

            # Checkpoints (écriture locale puis copie sur Drive)
            is_best = val_mae < best_val_mae
            best_val_mae = min(best_val_mae, val_mae)

            checkpoint = {
                "epoch": epoch + 1,
                "model_state_dict": faenet.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_mae": best_val_mae,
                "frame_averaging": faenet.frame_averaging,
                "use_weights": faenet.output_block.use_weights,
                "jump_connection": faenet.jump_connection,
                "target_idx": idx_regression_target,
                "target_mean": mu_target_train.detach().cpu(),
                "target_std": std_target_train.detach().cpu(),
                "sfa_generator_state": train_generator.get_state(),
                "shuffle_generator_state": shuffle_generator.get_state(),
                "torch_rng_state": torch.get_rng_state(),
            }

            torch.save(checkpoint, "/content/tmp_last.pt")
            shutil.copy("/content/tmp_last.pt", checkpoint_dir / "last.pt")

            if is_best:
                torch.save(checkpoint, "/content/tmp_best.pt")
                shutil.copy("/content/tmp_best.pt", checkpoint_dir / "best.pt")

        # Eval test sur best.pt
        checkpoint = torch.load(checkpoint_dir / "best.pt", map_location=device, weights_only=True)
        faenet.load_state_dict(checkpoint["model_state_dict"])
        faenet.frame_averaging = checkpoint["frame_averaging"]
        # faenet.frame_averaging = "full" check l'effet du full fa en inférence
        faenet.eval()

        # norm du training
        mu = checkpoint["target_mean"].to(device)
        std = checkpoint["target_std"].to(device)
        target_idx = checkpoint["target_idx"]

        test_generator = torch.Generator().manual_seed(1972)

        test_mae = 0.0
        n_samples = 0

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)

                _, y_pred_scaled = faenet(
                    batch, sfa_generator=test_generator
                )

                y_pred = mu + std * y_pred_scaled
                y_true = batch.y[:, target_idx:target_idx + 1]

                test_mae += (y_pred - y_true).abs().sum().item()
                n_samples += y_true.shape[0]

        test_mae /= n_samples

        summary = (
            f"[{mode}] DONE | Best epoch: {checkpoint['epoch']} | "
            f"Best Val MAE: {checkpoint['best_val_mae']:.6f} | Test MAE: {test_mae:.6f}"
        )
        print(summary)
        with open(log_path, "a") as f:
            f.write(summary + "\n")

    except Exception:
        err = traceback.format_exc()
        print(err)
        with open(checkpoint_dir / "error.txt", "w") as f:
            f.write(err)
    finally:
        for var in ["faenet", "optimizer", "scheduler", "train_loader", "val_loader", "test_loader"]:
            if var in locals():
                del locals()[var]
        torch.cuda.empty_cache()

# %% Extinction automatique du serveur Colab à la fin
import time
from google.colab import drive, runtime

time.sleep(30)  # laisse le temps à Drive de finir d'écrire les derniers fichiers
try:
    drive.flush_and_unmount()
except Exception as e:
    print("flush_and_unmount warning:", e)

runtime.unassign()  # supprime le serveur Colab